In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Goal: Predict sales for each row in test.csv
## unit(grain): one row=(date, store_nbr, family)
## features available at prediction time: onpromotions, oil price)
## output format: id, sales
## Metric: RMSLE

In [ ]:
df=pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv")
df['date']=pd.to_datetime(df['date'])
df

In [ ]:
test_df=pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/test.csv")
test_df['date']=pd.to_datetime(test_df['date'])
test_df

### train and test columns aligned

In [ ]:
keys=['date','store_nbr', 'family']

In [ ]:
(df['sales']<0).any()

### No negative sales

In [ ]:
df[keys].drop_duplicates().shape[0]==len(df)

In [ ]:
test_df[keys].drop_duplicates().shape[0]==len(test_df)

### number of rows align with keys columns in both

In [ ]:
sample_sub=pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv")
len(sample_sub)==len(test_df)

### sample submission has same number of rows as test dataset

## Loading other tables

In [ ]:
oil_data=pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/oil.csv")
oil_data

In [ ]:
oil_data['date'].nunique()

In [ ]:
oil_data['date']=pd.to_datetime(oil_data['date'])

### oil:
### key: date (len matches date count)
### grain: date

In [ ]:
stores=pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/stores.csv")
stores.head()

In [ ]:
stores['store_nbr'].nunique()==len(stores)

In [ ]:
stores.set_index('store_nbr', inplace=True)
stores.info()

### stores:
### key: store_nbr
### grain: store_nbr

In [ ]:
txn=pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/transactions.csv")
txn

In [ ]:
txn['date']=pd.to_datetime(txn['date'])

In [ ]:
txn[['date','store_nbr']].drop_duplicates().shape[0]==len(txn)

### transactions:
### key: date(not unique)
### grain: date, store_nbr

In [ ]:
holidays=pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv")
holidays

In [ ]:
holidays[['date','description']].drop_duplicates().shape[0]

In [ ]:
holidays['date']=pd.to_datetime(holidays['date'])

### holiday_events:
### key: date
### grain: date, description

## Combining test and train for same pipeline and lags for test to include train data (Eg: lags for 2017-08-16 will inlcude from train dates)

In [ ]:
# appending train and test data into master
master=pd.concat([df, test_df], ignore_index=True)
master.head(-5)

In [ ]:
# check duplicated rows in master based on 'date', 'store_nbr', 'family'
master.duplicated(keys).sum()==0

### master is without any duplicates

In [ ]:
master.info()

In [ ]:
# merge with stores
before_join=len(master)
master=master.join(stores, on='store_nbr',how='left')
after_join=len(master)
master

In [ ]:
before_join==after_join

In [ ]:
txn.info()

In [ ]:
txn[['date', 'store_nbr']].drop_duplicates().shape[0]

In [ ]:
master[['date', 'store_nbr']].drop_duplicates().shape[0]

In [ ]:
# merge with transactions
before_join=len(master)
master=pd.merge(master,txn, left_on=['date','store_nbr'],right_on=['date','store_nbr'],how='left')
after_join=len(master)
master

In [ ]:
before_join==after_join

In [ ]:
master = master.sort_values(["store_nbr", "date"]).copy()

# Make a flag BEFORE filling
master["tx_missing"] = master["transactions"].isna().astype(int)

# Carry last known traffic into future dates (test)
master["transactions"] = master.groupby("store_nbr")["transactions"].ffill()

# If anything still missing (early dates), fill with 0 or store median
master["transactions"] = master["transactions"].fillna(0.0)

In [ ]:
# merge with oil
before_join=len(master)
master=pd.merge(master,oil_data, left_on=['date'],right_on=['date'],how='left')
after_join=len(master)
master

In [ ]:
before_join==after_join

In [ ]:
# feature engineering holiday_events
# deleting tranferred days
del_count=(holidays['transferred']==True).sum()
before_delete=len(holidays)
holidays=holidays[holidays['transferred']==False].copy()
after_delete=len(holidays)
before_delete==(after_delete+del_count)

In [ ]:
holidays=holidays.drop(columns=['transferred'],axis=1,errors='ignore')

In [ ]:
holidays

In [ ]:
holidays['type'].unique()

In [ ]:
holidays['is_workday']=(holidays['type']=='Work Day').astype(int)

In [ ]:
holidays['is_event']=(holidays['type']=='Event').astype(int)
# holidays=holidays.drop(columns='event',axis=1)

In [ ]:
holidays['is_like_holiday']=(holidays['type'].isin(['Holiday', 'Transfer', 'Additional', 'Bridge'])).astype(int)

In [ ]:
store_date=master[['date','store_nbr','state','city']].drop_duplicates(['date','store_nbr']).copy()

In [ ]:
holidays['locale'].unique()

In [ ]:
hol_nat=(holidays[holidays['locale']=='National']).groupby('date', as_index=False).agg(
    holiday_national=('is_like_holiday','max'),
    event_national=('is_event','max'),
    workday_national=('is_workday','max'),
    nat_rows=('type','size')
)
store_date=store_date.merge(hol_nat,on='date',how='left')
store_date[store_date['holiday_national']==1]

In [ ]:
hol_local=(holidays[holidays['locale']=='Local']).groupby(['date','locale_name']).agg(
    holiday_local=('is_like_holiday','max'),
    event_local=('is_event','max'),
    workday_local=('is_workday','max'),
    local_rows=('type','size')
)
store_date=store_date.merge(hol_local,left_on=['date','city'], right_on=['date','locale_name'],how='left')
store_date[store_date['holiday_local']==1]

In [ ]:
# checking if local_name in holidays matches state name in store_date when locale='Regional'
set(holidays[holidays['locale']=='Regional']['locale_name'].unique())-set(store_date['state'].unique())

In [ ]:
hol_regional=(holidays[holidays['locale']=='Regional']).groupby(['date','locale_name'],as_index=False).agg(
    holiday_regional=('is_like_holiday','max'),
    event_regional=('is_event','max'),
    workday_regional=('is_workday','max'),
    regional_rows=('type','size')
)
hol_regional
store_date=store_date.merge(hol_regional,left_on=['date','state'],right_on=['date','locale_name'],how='left').drop(columns=['locale_name'])
store_date[store_date['holiday_regional']==1].head()

In [ ]:
store_date

In [ ]:
store_date.info()

In [ ]:
flag_cols=[c for c in store_date.columns if c.endswith(('_national','_local','regional'))]
count_cols=[c for c in store_date.columns if c.endswith('_rows')]

In [ ]:
# fill na with 0 in flag cols and count cols
store_date[flag_cols]=store_date[flag_cols].fillna(0).astype(int)
store_date[count_cols]=store_date[count_cols].fillna(0).astype(int)

In [ ]:
# merging store_date with master
before_join=len(master)
master=master.merge(store_date[['date','store_nbr']+flag_cols+count_cols], on=['date','store_nbr'], how='left')
after_join=len(master)

In [ ]:
after_join-before_join

### join of all tables complete

In [ ]:
master

In [ ]:
# Missing values handling
HORIZON=master.loc[master['sales'].isna(),'date'].nunique() # 16 in this case
test_dates=sorted(master.loc[master['sales'].isna(),'date'].unique())
print("test dates range:", test_dates[0]," to ",test_dates[-1])

In [ ]:
# sort values based on store_nbr, family, date
master=master.sort_values(['store_nbr','family','date']).reset_index(drop=True)
master['dcoilwtico'].isna().sum()

In [ ]:
# fill oil dcoilwtico(daily crude oil west texas intermediate Cushing Oklahoma) with forward fill date
master['oil_missing']=master['dcoilwtico'].isna().astype(int)

oil_series=(master[['date','dcoilwtico']]
           .drop_duplicates('date')
           .sort_values('date')
           .set_index('date')['dcoilwtico']
           .ffill()
           .bfill())

master=master.drop(columns=['dcoilwtico']).merge(oil_series.rename('dcoilwtico'), left_on='date',right_index=True, how='left')
master['dcoilwtico'].isna().sum()

In [ ]:
master['transactions'].isna().sum()
master['txn_missing']=master['transactions'].isna().astype(int)
master['transactions']=master['transactions'].fillna(0).astype(float)

In [ ]:
# building 3 rolling windows(folds) for validation
train_dates=sorted(master.loc[master['sales'].notna(),'date'].unique())

folds=[]
for k in [3,2,1]:
    val_end_idx=len(train_dates)-(k-1)*HORIZON
    val_start_idx=val_end_idx-HORIZON
    if val_start_idx<=0:
        continue
    val_dates=train_dates[val_start_idx:val_end_idx]
    cutoff=train_dates[val_start_idx-1]# last date in train set before these windows
    folds.append([cutoff,val_dates[0],val_dates[-1]])

for k in folds:
    print(k)

In [ ]:
def add_calendar_features(df):
    d = df["date"]
    df["dow"] = d.dt.dayofweek.astype("int8")         # 0=Mon
    df["weekofyear"] = d.dt.isocalendar().week.astype("int16")
    df["month"] = d.dt.month.astype("int8")
    df["day"] = d.dt.day.astype("int8")
    df["year"] = d.dt.year.astype("int16")
    df["is_weekend"] = (df["dow"] >= 5).astype("int8")
    df["is_month_start"] = d.dt.is_month_start.astype("int8")
    df["is_month_end"] = d.dt.is_month_end.astype("int8")
    return df

In [ ]:
# adding calendar features like week, month, year,....
master=add_calendar_features(master)

In [ ]:
# adding lags and rolling values of sales and onpromotions
def add_lag_rolling_features(df, lags=(1,7,14,28), rolls=(7,14,28)):
    group=df.groupby(['store_nbr','family'], sort=False)
    for L in lags:
        df[f'sales_lag_{L}']=group['sales'].shift(L)
        df[f'promo_lag_{L}']=group['onpromotion'].shift(L)
    for W in rolls:
        df[f'sales_rollmean_{W}']=group['sales'].transform(lambda s: s.shift(1)).rolling(W,min_periods=1).mean()
        df[f'sales_rollstd_{W}']=group['sales'].transform(lambda s: s.shift(1)).rolling(W,min_periods=2).std()#min_periods=2, since std for 1 is 0
    return df

In [ ]:
master=master.sort_values(['store_nbr','family','date']).reset_index(drop=True)
master=add_lag_rolling_features(master)

In [ ]:
# transform for RMSLE
master['y']=np.log1p(master['sales'])

In [ ]:
# there are few rows with lag_ values as NaN because they don't have previous 7 days or others, take example of fifth day in the series, it doesn't have last 7 days data yet
# we'll remove those rows that have nulls in these columns
feature_cols = [c for c in master.columns if c.startswith(("sales_lag_","sales_roll","promo_lag_"))] + ["onpromotion","transactions","dcoilwtico",
                "holiday_national","holiday_regional","holiday_local",
                "dow","weekofyear","month","day","is_weekend","is_month_start","is_month_end",
                "store_nbr","family","city","state","type","cluster"]
train_mask=(master['sales'].notna())
train_ready=master[train_mask].dropna(subset=[c for c in master.columns if c.startswith('sales_lag_')])
print("Train rows before:", train_mask.sum(), "after drop:", len(train_ready))

In [ ]:
master["date"] = pd.to_datetime(master["date"]).dt.normalize()

# Baselines

In [ ]:
# baseline 0(alias weekly naive)= weekly pattern
# baseline 0 helps in assuring that table joins, grain, lags, target are likely correct or something is broken.
# baseline is also for defining floor, model must be able to beat it
# lags_7 because retail often show strong weekly pattern
def rmsle(y_true,y_pred):
    y_true=np.maximum(y_true,0)# guardrails for negative values
    y_pred=np.maximum(y_pred,0)
    return np.sqrt(np.mean((np.log1p(y_pred)-np.log1p(y_true))**2))

In [ ]:
def baseline_lag7_score(master, val_start, val_end):
    df=master[(master['sales'].notna()) & (master['date']>=val_start) & (master['date']<=val_end)].copy()
    df=df.dropna(subset=['sales_lag_7'])
    pred=df['sales_lag_7']
    true=df['sales']
    return rmsle(true, pred)

In [ ]:
baseline_scores=[] # for each folds
for (train_end, val_start, val_end) in folds:
    baseline_scores.append(baseline_lag7_score(master, val_start, val_end))
    print(f"Fold {val_start.date()} to {val_end.date()}  |  Baseline lag_7 RMSLE: {baseline_scores[-1]:.5f}")

print("Mean Baseline lag_7 RMSLE:", float(np.mean(baseline_scores)))

In [ ]:
#baseline 1
required_cols=['sales_lag_14','sales_lag_28']
cat_cols = ["family", "city", "state", "type"]
for c in cat_cols:
    master[c]=master[c].astype('category')
lag_cols=[c for c in master.columns if c.startswith('sales_lag_')]
roll_cols=[c for c in master.columns if c.startswith(('sales_rollmean_','sales_rollstd_'))]
promo_cols=[c for c in master.columns if c.startswith('promo_lag_')]

calendar_cols = ["dow", "month", "weekofyear", "is_weekend"]
holiday_cols  = ["holiday_national", "holiday_regional", "holiday_local"]
other_num_cols = ["onpromotion", "transactions", "dcoilwtico", "oil_missing", "txn_missing", "store_nbr", "cluster"]

feature_cols=lag_cols + roll_cols + promo_cols + calendar_cols + holiday_cols + other_num_cols + cat_cols
print("Total features:", len(feature_cols))

In [ ]:
# fold data for train validation
def get_fold_data(master, train_end, val_start, val_end, feature_cols, required_cols):
    df=master[master['sales'].notna()].copy()
    train_df=df[df['date']<=train_end].dropna(subset=required_cols)
    val_df=df[(df['date']>=val_start) & (df['date']<=val_end)].dropna(subset=required_cols)
    
    X_tr=train_df[feature_cols]
    y_tr=train_df['y']
    X_va=val_df[feature_cols]
    y_va_sales=val_df['sales']
    return X_tr,y_tr,X_va,y_va_sales

In [ ]:
import lightgbm as lgb
SEED=42
def train_lgbm_and_score(master, train_end, val_start, val_end, feature_cols, required_cols):
    X_tr,y_tr,X_va,y_va_sales=get_fold_data(master,train_end,val_start,val_end,feature_cols,required_cols)
    model=lgb.LGBMRegressor(
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=127,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        objective='regression'
    )
    model.fit(
        X_tr,y_tr,
        eval_set=[(X_va,np.log1p(y_va_sales))],
        eval_metric='rmse',
        categorical_feature=[c for c in feature_cols if c in cat_cols],
        callbacks=[lgb.early_stopping(stopping_rounds=200,verbose=False)]
    )
    y_pred_log=model.predict(X_va)
    pred_sales=np.expm1(y_pred_log)
    pred_sales=np.maximum(pred_sales,0)
    return rmsle(y_va_sales.values,pred_sales)

### Baseline 1 without walk-forward

In [ ]:
# lgb_scores=[]
# for (train_end, val_start, val_end) in folds:
#     lgb_scores.append(train_lgbm_and_score(master, train_end, val_start, val_end, feature_cols, required_cols))
#     print(f"Fold {val_start.date()} to {val_end.date()}  |  LGBM RMSLE: {lgb_scores[-1]:.5f}")

# print("Mean LGBM RMSLE:", float(np.mean(lgb_scores)))

## but here lag 14 is used, which falls inside validations window(horizon=16). This cannot work because inside vaildation window the vlaues are null

In [ ]:
# walk forward approach to use predicted values when lag<HORIZON
LAGS_USED=[14,28]
required_cols=['sales_lag_14','sales_lag_28']

def build_lags_for_date(df, curr_date, lags):
    cur=df[df['date']==curr_date].copy()
    cur['row_id']=cur.index
    cur = cur.drop(columns=[f"sales_lag_{L}" for L in lags], errors="ignore")
    for lag in lags:
        prev_date=curr_date-pd.Timedelta(days=lag)
        prev=df[df['date']==prev_date][['store_nbr','family','sales_for_feat']].copy()
        prev=prev.rename(columns={'sales_for_feat':f'sales_lag_{lag}'})
        cur=cur.merge(prev,on=['store_nbr','family'], how='left', validate='many_to_one')#,suffixes=(None,None))
    return cur

In [ ]:
def walk_forward_fold_score(master, model, feature_cols, train_end, val_start, val_end, lags, debug=False):
    df=master[master['sales'].notna()].copy()

    df["date"] = pd.to_datetime(df["date"]).dt.normalize()

    train_end = pd.to_datetime(train_end).normalize()
    val_start = pd.to_datetime(val_start).normalize()
    val_end   = pd.to_datetime(val_end).normalize()

    df = df[df["date"] <= val_end].copy()
    # Hard check: validation window must exist in df
    n_val_rows = ((df["date"] >= val_start) & (df["date"] <= val_end)).sum()
    if debug:
        print("INSIDE: val rows =", n_val_rows)
    if n_val_rows == 0:
        raise ValueError(
            f"No rows found in validation window {val_start.date()} to {val_end.date()}. "
            f"Check date normalization and how folds were created."
        )
    
    df['sales_for_feat']=df['sales'].astype(float)
    val_dates = pd.date_range(val_start, val_end, freq="D")
    preds=[]
    y_true_all = []
    y_pred_all = []
    for d in val_dates:
        cur=build_lags_for_date(df,d,lags)
        # print(cur.columns)
        # cur=cur.dropna(subset=[f'sales_lag_{lag}' for lag in lags])
        
        if cur.empty:
            if debug: print("No rows for date:", d.date())
            # no rows for that date (shouldn't happen, but safe)
            continue
        for L in lags:
            col = f"sales_lag_{L}"
            if col not in cur.columns:
                cur[col] = np.nan
            cur[col] = cur[col].fillna(0.0)
        X=cur[feature_cols]
        if X.shape[0] == 0:
            if debug: print("X empty for date:", d.date())
            continue
        y_pred_log=model.predict(X)
        y_pred_sales=np.expm1(y_pred_log)
        y_pred_sales=np.maximum(y_pred_sales,0)
        df.loc[cur['row_id'],'sales_for_feat']=y_pred_sales

        y_true_all.append(df.loc[cur["row_id"], "sales"].values)
        y_pred_all.append(y_pred_sales)
        if debug:
            print(d.date(), "rows:", len(cur), "preds collected:", len(y_pred_sales))
        if len(y_true_all) == 0:
            raise ValueError(
            "Walk-forward produced 0 predictions. "
            "Most likely causes: df was filtered to date<=train_end, date mismatch, or val window not present."
            )
    # out=pd.concat(preds,ignore_index=True)
    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)
    # return rmsle(out['y_true'].values,out['y_pred'].values)
    return rmsle(y_true_all, y_pred_all)

In [ ]:
def train_model_for_fold(master, feature_cols, train_end, required_cols, cat_cols, seed=42):
    df=master[master['sales'].notna()].copy()
    train_df=df[df['date']<=train_end].dropna(subset=required_cols)
    X_tr=train_df[feature_cols]
    y_tr=train_df['y']
    model = lgb.LGBMRegressor(
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=127,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        objective="regression",
    )
    model.fit(X_tr,y_tr,categorical_feature=[c for c in feature_cols if c in cat_cols])
    return model

In [ ]:
H = master.loc[master['sales'].isna(), "date"].nunique()
train_dates = np.sort(master.loc[master['sales'].notna(), "date"].unique())

folds = []
for k in [3,2,1]:
    val_end = train_dates[-(k-1)*H - 1]
    val_start = train_dates[-(k-1)*H - H]
    train_end = train_dates[-(k-1)*H - H - 1]
    folds.append((train_end, val_start, val_end))

print(folds)

In [ ]:
folds = [(pd.to_datetime(a).normalize(),
          pd.to_datetime(b).normalize(),
          pd.to_datetime(c).normalize()) for (a,b,c) in folds]
folds

In [ ]:
#test code
fold_count=1
(train_end, val_start, val_end) = folds[fold_count]

print(f"fold[{fold_count}]:", train_end, val_start, val_end)
print("types:", type(train_end), type(val_start), type(val_end))

# Ensure master date is clean
print("master date dtype:", master["date"].dtype)
print("train min/max:",
      master.loc[master['sales'].notna(), "date"].min(),
      master.loc[master['sales'].notna(), "date"].max())

# Do we even have rows for the validation window?
mask = (master['sales'].notna()) & (master["date"]>=val_start) & (master["date"]<=val_end)
print("rows in val window:", mask.sum())

# Check if a single day exists
print("rows on val_start:", ((master['sales'].notna()) & (master["date"]==val_start)).sum())

In [ ]:
# walk_scores=[]
# for (train_end,val_start,val_end) in folds:
#     model=train_model_for_fold(master,feature_cols,train_end,required_cols,cat_cols)
#     sc=walk_forward_fold_score(master,model,feature_cols,train_end,val_start,val_end,LAGS_USED)
#     walk_scores.append(sc)

# print("Mean Walk-forward RMSLE:", float(np.mean(walk_scores)))

In [ ]:
master['date']=pd.to_datetime(master['date'].dt.normalize())
for c in cat_cols:
    master[c]=master[c].astype('category')

In [ ]:
keep_sales = {"sales_lag_14", "sales_lag_28"}

feature_cols_fixed = [
    c for c in feature_cols
    if (not c.startswith("sales_")) or (c in keep_sales)
]

In [ ]:
required_cols=[f'sales_lag_{L}' for L in LAGS_USED]
train_df=master[master['sales'].notna()].copy()
train_df=train_df.dropna(subset=required_cols)

X_tr=train_df[feature_cols_fixed]
y_tr=train_df['y']

final_model = lgb.LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=127,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="regression",
)

final_model.fit(X_tr, y_tr, categorical_feature=[c for c in feature_cols if c in cat_cols])

In [ ]:
def walk_forward_predict_test(master, model, feature_cols, lags, debug=False):
    work=master.copy()
    work['date']=pd.to_datetime(work['date']).dt.normalize()
    work['sales_for_feat']=work['sales'].astype(float)

    test_dates=np.sort(work.loc[work['sales'].isna(),'date'].unique())
    preds=[]
    for d in test_dates:
        cur=build_lags_for_date(work,d,lags)
        if cur.empty:
            if debug: print('No rows for test date:',d)
            continue
        for L in lags:
            col=f'sales_lag_{L}'
            if col not in cur.columns:
                cur[col]=np.nan
            cur[col]=cur[col].fillna(0.0)
    
        X=cur[feature_cols]
        y_pred_log=model.predict(X)
        y_pred_sales=np.maximum(np.expm1(y_pred_log),0)
    
        work.loc[cur['row_id'],'sales_for_feat']=y_pred_sales
    
        preds.append(pd.DataFrame({
                "id": cur["id"].values,
                "sales": y_pred_sales
            }))
        if debug:
            print(pd.to_datetime(d).date(), "rows:", len(cur))

    return pd.concat(preds, ignore_index=True)

In [ ]:
test_pred = walk_forward_predict_test(master, final_model, feature_cols_fixed, LAGS_USED, debug=True)
test_pred = test_pred.sort_values("id").reset_index(drop=True)
test_pred.head()

In [ ]:
test_pred['sales']=round(test_pred['sales'],1)
n_test = (master["sales"].isna()).sum()
print("Expected test rows:", n_test)
print("Predicted rows:", len(test_pred))
print("Duplicate ids:", test_pred["id"].duplicated().sum())

test_pred.to_csv("submission.csv", index=False)
print("Saved submission.csv")

In [ ]:
test_pred['sales'].min()

In [ ]:
test_pred['sales'].isna().sum()

### No empty predictions => all test values are predicted

In [ ]:
test_pred['sales'].describe(percentiles=[0.5,0.9,0.99]).to_frame()

In [ ]:
sub = pd.read_csv("submission.csv")
print(sub.head())
print(sub["sales"].describe())
print(sub["sales"].quantile([0.5, 0.9, 0.99, 1.0]))
print("NaN:", sub["sales"].isna().sum(), " Min:", sub["sales"].min())

In [ ]:
# master[master['sales'].notna()]['sales'].max()
print(master[master['sales'].notna()][["sales"]].quantile([0.5, 0.9, 0.99, 1.0]))

In [ ]:
train_sales = master.loc[master["sales"].notna(), "sales"]
print("TRAIN sales stats:")
print(train_sales.describe(percentiles=[0.5,0.9,0.99]))

print("\nSUBMISSION sales stats:")
print(sub["sales"].describe(percentiles=[0.5,0.9,0.99]))

In [ ]:
test_mask = master["sales"].isna()
train_mask = master["sales"].notna()

cols_to_check = ["transactions", "dcoilwtico", "onpromotion"]
print("Missing rate in TEST:\n", master.loc[test_mask, cols_to_check].isna().mean())
print("\nTEST describe:\n", master.loc[test_mask, cols_to_check].describe())
print("\nTRAIN describe:\n", master.loc[train_mask, cols_to_check].describe())

In [ ]:
# work = master.copy()
# work["date"] = pd.to_datetime(work["date"]).dt.normalize()
# work["sales_for_feat"] = work["sales"].astype(float)

# preds=[]
# for d in np.sort(work.loc[work["sales"].isna(), "date"].unique()):
#     cur = build_lags_for_date(work, d, [28])
#     cur["sales_lag_28"] = cur["sales_lag_28"].fillna(0.0)
#     preds.append(pd.DataFrame({"id": cur["id"].values, "sales": cur["sales_lag_28"].values}))

# subA = pd.concat(preds, ignore_index=True).sort_values("id")
# subA.to_csv("submission.csv", index=False)
# print(subA["sales"].describe(percentiles=[0.5,0.9,0.99,1.0]))